In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import MNIST

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5))
])

train_set = MNIST(root="./data",download=True,train=True,transform=transform)
test_set = MNIST(root="./data",download=True,train=False,transform=transform)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:01<00:00, 5.19MB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:01<00:00, 1.11MB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 6.71MB/s]


In [3]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
test_loader = DataLoader(test_set,batch_size=64)

## **Build the RNN**

In [4]:
class RNN(nn.Module):
  def __init__(self,input_size=28,hidden_size=128,num_layers=1,num_classes = 10):   #28*28 image
    super().__init__()
    self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
    self.fc = nn.Linear(hidden_size,num_classes)

  def forward(self,x):
    out,_ = self.rnn(x)
    out = self.fc(out[:,-1,:])
    return out

In [5]:
model = RNN()

criterion = nn.CrossEntropyLoss()   # automatically applies softmax activation fucntion
optimizer = optim.Adam(model.parameters())

# **Train the RNN**

In [6]:
epochs = 10
for epoch in range(epochs):
  model.train()
  epoch_training_loss = 0.0
  for Xb,yb in train_loader:
    optimizer.zero_grad()
    Xb = Xb.squeeze(1) #[batch,1,28,28] => [batch,28,28]
    outputs = model(Xb)
    loss = criterion(outputs,yb)
    loss.backward()
    optimizer.step()
    epoch_training_loss += loss.item()

  print(f"Epoch : {epoch+1} and loss : {epoch_training_loss/len(train_loader)}")


Epoch : 1 and loss : 0.752652263765269
Epoch : 2 and loss : 0.28255997193472854
Epoch : 3 and loss : 0.21960611357641563
Epoch : 4 and loss : 0.18342845775941605
Epoch : 5 and loss : 0.16271534387165232
Epoch : 6 and loss : 0.15153872981163136
Epoch : 7 and loss : 0.1429782365875712
Epoch : 8 and loss : 0.13130592476965777
Epoch : 9 and loss : 0.13650177658668586
Epoch : 10 and loss : 0.1225965291679676


# **Evaluate the Model**

In [7]:
model.eval()
correct_vals = 0
total_vals = 0
with torch.no_grad():
  for Xb,yb in test_loader:
    Xb = Xb.squeeze(1)
    outputs = model(Xb)
    _,predicted = torch.max(outputs,1)
    correct_vals += (predicted == yb).sum().item()
    total_vals += yb.size(0)

  print(f"Accuracy : {(correct_vals/total_vals)*100}")

Accuracy : 96.36
